# Apresentação: Pipeline de Features
Este notebook carrega e apresenta os resultados gerados pelo pipeline de extração de features.
Paths usados assumem o ambiente do compose: `/workspace/data/output/features.parquet`.

In [ ]:
# Imports e configurações iniciais
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
sns.set(style="whitegrid")
%matplotlib inline


In [ ]:
# Parâmetros de caminho
IMAGES_DIR = '/workspace/data/images'
FEATURES_PARQUET = '/workspace/data/output/features.parquet'
METADATA_PATH = '/workspace/data/metadata_consolidated.csv'
print('IMAGES_DIR =', IMAGES_DIR)
print('FEATURES_PARQUET =', FEATURES_PARQUET)


## Checagens rápidas

In [ ]:
# Verificar existência dos arquivos esperados
print('Images dir exists:', os.path.isdir(IMAGES_DIR))
print('Features parquet exists:', os.path.isfile(FEATURES_PARQUET))
print('Metadata exists:', os.path.isfile(METADATA_PATH))
# listar algumas imagens
if os.path.isdir(IMAGES_DIR):
    imgs = os.listdir(IMAGES_DIR)[:10]
    print('Exemplos de imagens:', imgs)


## Carregar features.parquet

In [ ]:
# Carregar com pandas (pyarrow deve estar instalado na imagem)
try:
    df = pd.read_parquet(FEATURES_PARQUET)
    print('Linhas:', len(df), 'Imagens únicas:', df['image_id'].nunique() if 'image_id' in df.columns else 'n/a')
    display(df.head())
except Exception as e:
    print('Erro ao carregar parquet:', e)


## Resumo estatístico e contagens

In [ ]:
# Contagens por tipo de detecção (assume coluna 'detection_type')
if 'detection_type' in df.columns:
    counts = df['detection_type'].value_counts()
    print(counts)
    plt.figure(figsize=(6,4))
    sns.barplot(x=counts.index, y=counts.values)
    plt.title('Detecções por tipo')
    plt.ylabel('count')
    plt.xlabel('detection_type')
    plt.show()
else:
    print('Coluna detection_type não encontrada')


## Amostras visuais: imagens com caixas e labels

In [ ]:
from matplotlib.patches import Rectangle
def show_image_with_boxes(image_path, detections):
    img = Image.open(image_path).convert('RGB')
    plt.figure(figsize=(6,6))
    fig,ax = plt.subplots(1,1,figsize=(6,6))
    ax.imshow(img)
    for det in detections:
        if not isinstance(det, dict):
            continue
        bbox = det.get('bbox') or det.get('box')
        label = det.get('label') or det.get('detection_type') or ''
        if bbox is None:
            continue
        # bbox esperado: [x1,y1,x2,y2] ou [x,y,w,h]
        if len(bbox)==4:
            x1,y1,x2,y2 = bbox
            w = x2-x1
            h = y2-y1
        else:
            x1,y1,w,h = bbox
        rect = Rectangle((x1,y1), w, h, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, max(0,y1-6), str(label), color='yellow', fontsize=8, backgroundcolor='black')
    plt.axis('off')
    plt.show()

# Mostrar até 6 imagens com suas detecções
sample_ids = df['image_id'].drop_duplicates().tolist()[:6] if 'image_id' in df.columns else []
for img_id in sample_ids:
    row_dets = df[df['image_id']==img_id].to_dict(orient='records')
    img_path = os.path.join(IMAGES_DIR, img_id) if os.path.isfile(os.path.join(IMAGES_DIR, img_id)) else None
    if img_path:
        print('Image:', img_id)
        show_image_with_boxes(img_path, row_dets)
    else:
        print('Imagem não encontrada em IMAGES_DIR:', img_id)


## OCR: tabela de exemplos

In [ ]:
# Mostrar exemplos de OCR (se coluna 'ocr_text' existir)
if 'ocr_text' in df.columns:
    ocr_df = df[['image_id','ocr_text','ocr_confidence','bbox']].dropna(subset=['ocr_text']).head(20)
    display(ocr_df)
else:
    print('Coluna ocr_text não encontrada')


## Embeddings: redução de dimensionalidade e visualização

In [ ]:
# Converter coluna 'embedding' para matriz e rodar PCA + TSNE (se disponível)
if 'embedding' in df.columns:
    embs = np.stack(df['embedding'].apply(lambda x: np.array(x, dtype=np.float32)))
    print('embs shape', embs.shape)
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    pca = PCA(n_components=50)
    embs_pca = pca.fit_transform(embs)
    tsne = TSNE(n_components=2, init='pca', random_state=42, perplexity=30)
    embs_2d = TSNE(n_components=2, init='pca', random_state=42, perplexity=30).fit_transform(embs_pca)
    plot_df = pd.DataFrame({'x':embs_2d[:,0],'y':embs_2d[:,1]})
    if 'detection_type' in df.columns:
        plot_df['type'] = df['detection_type'].values
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=plot_df, x='x', y='y', hue='type' if 'detection_type' in plot_df.columns else None, s=10, alpha=0.7)
    plt.title('Embeddings (PCA50 -> t-SNE)')
    plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
    plt.show()
else:
    print('Coluna embedding não encontrada')


## Busca semântica de exemplo (nearest neighbors)

In [ ]:
# Demonstração de busca por texto usando sentence-transformers (se instalado)
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    def semantic_search(query, topk=5):
        qv = model.encode([query])[0] / np.linalg.norm(model.encode([query])[0])
        E = embs / np.linalg.norm(embs, axis=1, keepdims=True)
        sims = (E @ qv).astype(np.float32)
        idx = np.argsort(-sims)[:topk]
        return idx, sims[idx]
    # exemplo
    if 'embedding' in df.columns:
        idxs, scores = semantic_search('exemplo de teste')
        print('Top resultados:')
        display(df.iloc[idxs][['image_id','ocr_text']])
except Exception as e:
    print('Não foi possível executar busca semântica:', e)


## Próximos passos / exportação

In [ ]:
# Exportar subset para anotação humana
out_csv = '/workspace/data/output/text_samples_for_annotation.csv'
if 'ocr_text' in df.columns:
    df[df['detection_type']=='text'][['image_id','ocr_text','ocr_confidence']].head(500).to_csv(out_csv, index=False)
    print('Exportado', out_csv)
else:
    print('Nada exportado (ocr_text não disponível)')
